# 📊 Stock Fundamentals CEO Dashboard
**Run all cells from top to bottom. A CEO Report will open in your browser at the end.**

Only one file to run: this notebook. All heavy logic lives in `src/`.

---
## Cell 1 — Configuration (edit this cell)
This is the only cell you need to edit. Fill in your stock and settings below.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit everything in this cell, then Run All
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. Stock ticker(s) ────────────────────────────────────────────────────────
TICKER = "KO"           # Primary ticker to analyze
PEER_TICKERS = []         # Optional list of peer tickers, e.g. ["MSFT", "GOOG"]
                          # Leave as [] to skip peer comparison

# ── 2. Ownership mode ─────────────────────────────────────────────────────────
# "new"      → New Position Analysis  (recommendation: Strong Buy / Buy / Hold / Avoid)
# "existing" → Existing Holding       (recommendation: Buy More / Hold / Trim / Sell)
OWNERSHIP_MODE = "existing" 

# ── 3. Existing Holding details (only used when OWNERSHIP_MODE = "existing") ──
PURCHASE_DATE     = "null"   # Date you bought (YYYY-MM-DD)
COST_BASIS        = None         # Your average cost per share (USD)
SHARES_OWNED      = None            # Number of shares you own
TOTAL_PORTFOLIO   = None        # Total portfolio value in USD (for position weight)

# ── 3a. Allocation targets (only used when OWNERSHIP_MODE = "existing") ───────
TARGET_ALLOCATION  = 0.05   # 5%  — ideal / target weight for this position
MAX_ALLOCATION     = 0.10   # 10% — upper bound before trimming is considered
SEVERE_OVERWEIGHT  = 0.15   # 15% — trigger strong trim / sell pressure

# ── 4. Threshold overrides (optional) ─────────────────────────────────────────
# Override any default threshold by adding key: value pairs here.
# Keys use dot-notation matching config/default_thresholds.json.
# Example: {"valuation.pe_cheap": 20, "growth.revenue_cagr_1yr_strong": 0.20}
# Leave as {} to use smart sector-adjusted defaults.
THRESHOLD_OVERRIDES = {}


# ── 5. Saved threshold profile (optional) ─────────────────────────────────────
# Name of a previously saved threshold set (created with tm.save("my_profile")).
# Set to None to use defaults + overrides above.
THRESHOLD_PROFILE = None   # e.g. "growth_investor"

# ── 6. ML predictions panel ───────────────────────────────────────────────────
# Adds a panel with estimated fair-value range and outperformance probability.
# Clearly labeled as predictions — requires scikit-learn.
ENABLE_ML_PREDICTIONS = True

# ── 7. Notes (optional) ───────────────────────────────────────────────────────
# These will appear in the CEO Report as analyst notes.
NOTES = """
Add your qualitative notes here. Why do you like this company? What risks concern you?
What would change your thesis? These notes appear verbatim in the CEO Report.
"""

# ── 8. Options ────────────────────────────────────────────────────────────────
FORCE_REFRESH = False      # Set True to bypass cache and re-fetch all data
OPEN_REPORT   = True       # Auto-open the HTML report in your browser when done

# ── 9. Report tab/view ───────────────────────────────────────────────────────
# "present" → Classic 6-question CEO dashboard (current framework)
# "past"    → First-principles P1-P6 only
# "future"  → First-principles F1-F6 only
# You can either set REPORT_TAB manually or use the interactive selector below.
REPORT_TAB = "past"
FIRST_PRINCIPLES_AS_OF = None   # Optional YYYY-MM-DD for strict PIT cutoff

# Interactive selector (shows directly in Cell 1)
try:
    import ipywidgets as widgets
    from IPython.display import display
    _report_tab_selector = widgets.ToggleButtons(
        options=["present", "past", "future"],
        value=REPORT_TAB,
        description="Report Tab:",
        button_style="",
        tooltips=[
            "Classic CEO dashboard",
            "First-principles Past (P1-P6)",
            "First-principles Future (F1-F6)",
        ],
    )
    def _sync_report_tab(change):
        globals()["REPORT_TAB"] = change["new"]
    _report_tab_selector.observe(_sync_report_tab, names="value")
    display(_report_tab_selector)
    REPORT_TAB = _report_tab_selector.value
except Exception as e:
    print(f"Report Tab widget unavailable ({e}). Using REPORT_TAB variable directly.")

print(f"Configuration loaded: {TICKER} | Mode: {OWNERSHIP_MODE} | ML: {ENABLE_ML_PREDICTIONS} | Tab: {REPORT_TAB}")


ToggleButtons(description='Report Tab:', index=1, options=('present', 'past', 'future'), tooltips=('Classic CE…

Configuration loaded: KO | Mode: existing | ML: True | Tab: past


---
## Cell 2 — Setup & Imports

In [2]:

# Auto-reload: picks up any changes to src/ without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Add project root to path so 'src' is importable
CWD = Path(os.getcwd())
if (CWD / "src").exists() and str(CWD) not in sys.path:
    sys.path.insert(0, str(CWD))
elif (CWD.parent / "src").exists() and str(CWD.parent) not in sys.path:
    sys.path.insert(0, str(CWD.parent))

from src.data_fetcher       import StockData, fetch_multiple
from src.metrics_calculator import MetricsCalculator
from src.threshold_manager  import ThresholdManager
from src.visualizations     import build_all_charts, price_history_chart, dividend_history_chart
from src.panel_generator    import PanelGenerator
from src.report_generator   import generate_report
from src.first_principles   import FirstPrinciplesEngine
from src.utils              import parse_date, fmt_currency, fmt_pct

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # Use 'browser' if charts don't render

print("✅ Imports successful")


✅ Imports successful


---
## Cell 3 — Fetch Data

In [3]:
print(f"Fetching data for {TICKER}...")
stock = StockData(TICKER, force_refresh=FORCE_REFRESH)
print(f"✅ {stock.staleness_label()}")

# Fetch peer data if requested
peer_stocks = {}
if PEER_TICKERS:
    print(f"Fetching peer data for: {PEER_TICKERS}")
    peer_stocks = fetch_multiple(PEER_TICKERS, force_refresh=FORCE_REFRESH)
    print(f"✅ Peers loaded: {list(peer_stocks.keys())}")

2026-05-27 10:33:30 [INFO] dashboard.data_fetcher — Loading data for KO…
/Users/friday/SynologyDrive/Personal Drive/stock programs/buy_hold_sell_dashboard/src/data_fetcher.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
/Users/friday/SynologyDrive/Personal Drive/stock programs/buy_hold_sell_dashboard/src/data_fetcher.py:296: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df.columns = pd.to_datetime(df.columns, errors="coerce")
2026-05-27 10:33:30 [INFO] dashboard.data_fetcher — Data loaded for KO — Data as of 2026-05-27 10:33 PDT — last filing 147 days ago


Fetching data for KO...
✅ Data as of 2026-05-27 10:33 PDT — last filing 147 days ago


---
## Cell 4 — Compute Metrics

In [4]:
# Parse existing holding details (only used if mode = "existing")
purchase_date_parsed = parse_date(PURCHASE_DATE) if OWNERSHIP_MODE == "existing" else None
cost_basis_val       = float(COST_BASIS)         if OWNERSHIP_MODE == "existing" else None
shares_val           = float(SHARES_OWNED)       if OWNERSHIP_MODE == "existing" else None
portfolio_val        = float(TOTAL_PORTFOLIO)    if OWNERSHIP_MODE == "existing" else None

# Build metrics
calc = MetricsCalculator(
    stock=stock,
    mode=OWNERSHIP_MODE,
    purchase_date=purchase_date_parsed,
    cost_basis=cost_basis_val,
    shares_owned=shares_val,
    total_portfolio_value=portfolio_val,
    notes=NOTES,
)
metrics = calc.compute()

# Compute peer metrics
peer_metrics = {}
for t, s in peer_stocks.items():
    peer_calc = MetricsCalculator(stock=s, mode="new")
    peer_metrics[t] = peer_calc.compute()

print(f"✅ Metrics computed for {metrics.ticker} — {metrics.company_name}")
print(f"   Overall Score: {metrics.overall_score:.1f}/10")
print(f"   Sector: {metrics.q1.sector} | {metrics.q1.cap_category}")
print(f"   TTM Revenue: {fmt_currency(metrics.q1.ttm_revenue)}")
print(f"   Operating Margin: {fmt_pct(metrics.q3.operating_margin)}")
print(f"   FCF Yield: {fmt_pct(metrics.q4.fcf_yield)}")
print(f"   P/E: {metrics.q6.pe_ratio}")

TypeError: float() argument must be a string or a real number, not 'NoneType'

---
## Cell 5 — Load Thresholds

In [ ]:
# Load threshold manager with sector awareness
if THRESHOLD_PROFILE:
    try:
        tm = ThresholdManager.load(THRESHOLD_PROFILE)
        print(f"✅ Loaded threshold profile: {THRESHOLD_PROFILE}")
    except FileNotFoundError:
        print(f"⚠️ Profile '{THRESHOLD_PROFILE}' not found — using defaults")
        tm = ThresholdManager(sector=metrics.q1.sector)
else:
    tm = ThresholdManager(sector=metrics.q1.sector)

# Apply user overrides
if THRESHOLD_OVERRIDES:
    tm.override_many(THRESHOLD_OVERRIDES)
    print(f"✅ Applied {len(THRESHOLD_OVERRIDES)} threshold overrides")

# Display effective thresholds
print()
print(tm.summary())

# Optionally save this profile for future use:
# tm.save("my_profile")


Thresholds — sector: Technology, user overrides: 0

  GROWTH
    Revenue CAGR (strong):    15%
    Revenue CAGR (acceptable):5%

  PROFITABILITY
    Operating margin (strong):    20%
    Operating margin (acceptable):8%
    ROIC (strong):                15%

  CASH FLOW
    FCF yield (strong):     5%
    FCF margin (strong):    15%

  BALANCE SHEET
    Debt/EBITDA (strong):   1.5x
    Debt/EBITDA (danger):   5.0x
    Current ratio (strong): 2.0

  VALUATION
    P/E cheap: 24.0, fair: 40.0, expensive: 56.0
    EV/EBITDA cheap: 10, fair: 18, expensive: 25


---
## Cell 6 — Build Charts

In [ ]:
# Attach raw data that visualizations need (price history, dividends)
# These are stored directly on the metrics object for downstream access
metrics._history_df = stock.history
metrics._divs_series = stock.dividends

# Build all charts
charts = build_all_charts(metrics, tm)

# Build the price history chart with our details
charts["price_history"] = price_history_chart(
    history=stock.history,
    ticker=metrics.ticker,
    purchase_date=purchase_date_parsed if OWNERSHIP_MODE == "existing" else None,
    cost_basis=cost_basis_val if OWNERSHIP_MODE == "existing" else None,
    watermark=metrics.staleness_label,
)

print(f"✅ Built {len(charts)} charts")

✅ Built 15 charts


---
## Cell 7 — Generate Panels

In [ ]:
# Initialize panel generator
pg = PanelGenerator(
    metrics=metrics,
    tm=tm,
    target_allocation=TARGET_ALLOCATION if OWNERSHIP_MODE == "existing" else None,
    max_allocation=MAX_ALLOCATION if OWNERSHIP_MODE == "existing" else None,
    severe_overweight=SEVERE_OVERWEIGHT if OWNERSHIP_MODE == "existing" else None,
)

# Generate all panels
panel_bullish   = pg.panel_bullish()
panel_hold      = pg.panel_hold()
panel_bearish   = pg.panel_bearish()
panel_dividends = pg.panel_dividends(dividend_series=stock.dividends)
panel_portfolio = pg.panel_portfolio_risk()   # None if mode != "existing"

# ML predictions panel (optional)
panel_ml = pg.panel_ml_predictions() if ENABLE_ML_PREDICTIONS else None

# Final recommendation — returns (rec, explanation, audit_dict)
recommendation, rec_explanation, rec_audit = pg.final_recommendation(
    bullish=panel_bullish,
    hold=panel_hold,
    bearish=panel_bearish,
)

# Decision Audit panel (structured grid showing exactly why this action was chosen)
panel_audit = pg.panel_decision_audit(rec_audit)

print(f"\n{'='*60}")
print(f"  RECOMMENDATION: {recommendation}")
print(f"{'='*60}")
print(f"  {rec_explanation}")
print(f"{'='*60}\n")

print(f"  Company:          {rec_audit['company_verdict']}")
print(f"  Valuation:        {rec_audit['valuation_verdict']}")
if OWNERSHIP_MODE == "existing":
    print(f"  Portfolio Weight: {rec_audit.get('weight_verdict', 'N/A')}")
    print(f"  Gain/Loss:        {rec_audit.get('gain_loss_verdict', 'N/A')}")
print(f"  Primary Driver:   {rec_audit['primary_driver']}")
print(f"  Secondary Driver: {rec_audit['secondary_driver']}")
print(f"\nBullish signals: {panel_bullish.signal_count}")
print(f"Bearish signals: {panel_bearish.signal_count}")


TypeError: PanelGenerator.__init__() got an unexpected keyword argument 'target_allocation'

---
## Cell 8 — Preview Charts Inline
*(Optional — run to view interactive charts here in the notebook)*

In [ ]:
# Preview key charts inline
charts["price_history"].show()
charts["scorecard_radar"].show()
charts["q3_margins"].show()
charts["q4_fcf_yield_gauge"].show()
charts["q5_debt_ebitda_gauge"].show()
charts["q6_pe_gauge"].show()
charts["q2_revenue_cagr"].show()

---
## Cell 9 — Preview Panel Bullets Inline

In [ ]:
from IPython.display import Markdown, display

def show_panel(panel, emoji=""):
    if panel is None:
        return
    display(Markdown(f"### {emoji} {panel.title}"))
    for b in panel.bullets:
        if b:
            display(Markdown(f"- {b}"))
    print()

show_panel(panel_audit,     "🔍")
show_panel(panel_bullish,   "✅")
show_panel(panel_hold,      "⏸️")
show_panel(panel_bearish,   "⚠️")
show_panel(panel_dividends, "💰")
show_panel(panel_portfolio, "📁")
if panel_ml:
    show_panel(panel_ml,    "🤖")


### ✅ Why Buy

- High operating margin: 22.7% (above strong threshold of 20.0%)

- High FCF yield: 11.1% — the business generates substantial free cash.

- Excellent FCF margin: 15.4%.

- Cheap P/E: 8.3x (below cheap threshold of 15x)

...

### 🤖 Forward Signals (ML Predictions)

- ⚠️  ALL figures in this panel are MODEL ESTIMATES based on historical patterns. They are NOT guaranteed and should NOT be used as the sole basis for any investment decision.

- Estimated fair-value range: $17.99 – $21.73 (current price: $25.23)

- Implied downside to midpoint: -21.3% (PREDICTION — not guaranteed)

- Features used: roic, fcf_yield, revenue_1yr_cagr, operating_margin, debt_to_ebitda, pe_ratio, ev_ebitda, price_to_sales

---
## Cell 10 — Generate CEO Report ← **Run this last**
This generates the HTML report and saves it to `outputs/`. It will auto-open in your browser.

In [ ]:
# Normalize report tab selection
REPORT_TAB = str(REPORT_TAB).strip().lower()
if REPORT_TAB not in {"present", "past", "future"}:
    raise ValueError("REPORT_TAB must be one of: 'present', 'past', 'future'.")

from src.utils import log_session
from IPython.display import Markdown, display
import traceback
import webbrowser

if REPORT_TAB == "present":
    # Collect all panels
    panels = {
        "decision_audit": panel_audit,
        "bullish":        panel_bullish,
        "hold":           panel_hold,
        "bearish":        panel_bearish,
        "dividends":      panel_dividends,
    }
    if panel_portfolio:
        panels["portfolio"] = panel_portfolio
    if panel_ml and ENABLE_ML_PREDICTIONS:
        panels["ml"] = panel_ml

    # Generate and save the CEO Report
    report_path = generate_report(
        metrics=metrics,
        panels=panels,
        recommendation=recommendation,
        recommendation_explanation=rec_explanation,
        charts=charts,
        open_in_browser=OPEN_REPORT,
        peer_metrics=peer_metrics if peer_metrics else None,
        notes=NOTES,
    )

    # Log this session
    log_session(
        ticker=metrics.ticker,
        mode=OWNERSHIP_MODE,
        recommendation=recommendation,
        notes=NOTES[:100] if NOTES else "",
    )

    display(Markdown(f"""
---
## ✅ CEO Report Complete

**Recommendation: {recommendation}**

{rec_explanation}

Report saved to: `{report_path}`
"""))

else:
    try:
        fp_as_of = parse_date(FIRST_PRINCIPLES_AS_OF) if FIRST_PRINCIPLES_AS_OF else None
        if FIRST_PRINCIPLES_AS_OF and fp_as_of is None:
            raise ValueError("FIRST_PRINCIPLES_AS_OF must be a valid date (YYYY-MM-DD, MM/DD/YYYY, DD/MM/YYYY, or YYYYMMDD).")

        fp_engine = FirstPrinciplesEngine(force_refresh=FORCE_REFRESH)
        try:
            fp_report = fp_engine.run(TICKER, as_of_date=fp_as_of)
        except Exception as primary_exc:
            if FORCE_REFRESH:
                print(f"First-principles run failed with FORCE_REFRESH=True ({primary_exc}). Retrying with cache fallback...")
                fp_engine = FirstPrinciplesEngine(force_refresh=False)
                fp_report = fp_engine.run(TICKER, as_of_date=fp_as_of)
            else:
                raise

        SAVE_FIRST_PRINCIPLES_JSON = bool(globals().get("SAVE_FIRST_PRINCIPLES_JSON", True))
        fp_json_path = fp_engine.save_json_report(fp_report, time_view=REPORT_TAB) if SAVE_FIRST_PRINCIPLES_JSON else None

        fp_recommendation = fp_report.synthesis.decision
        fp_explanation = (
            f"{REPORT_TAB.title()} first-principles signal mix. "
            f"Strong {fp_report.synthesis.strong_count}, "
            f"Watch {fp_report.synthesis.watch_count}, "
            f"Weak {fp_report.synthesis.weak_count}, "
            f"Insufficient {fp_report.synthesis.insufficient_count}."
        )

        report_path = generate_report(
            metrics=metrics,
            panels={},
            recommendation=fp_recommendation,
            recommendation_explanation=fp_explanation,
            charts={},
            open_in_browser=OPEN_REPORT,
            peer_metrics=None,
            notes=NOTES,
            report_view=REPORT_TAB,
            first_principles_report=fp_report,
        )

        log_session(
            ticker=TICKER,
            mode=f"first_principles_{REPORT_TAB}",
            recommendation=fp_report.synthesis.decision,
            notes=NOTES[:100] if NOTES else "",
        )

        display(Markdown(f"""
---
## ✅ CEO-Style First-Principles {REPORT_TAB.title()} Report Complete

**Decision: {fp_report.synthesis.decision}**

Signals: Strong **{fp_report.synthesis.strong_count}** | Watch **{fp_report.synthesis.watch_count}** | Weak **{fp_report.synthesis.weak_count}** | Insufficient **{fp_report.synthesis.insufficient_count}**

Report saved to: `{report_path}`
{f"\nJSON audit saved to: `{fp_json_path}`\n" if fp_json_path else ""}
"""))
    except Exception as e:
        print("❌ First-principles report generation failed. Full traceback below:")
        traceback.print_exc()
        raise


2026-05-12 21:11:41 [INFO] dashboard.report — CEO Report saved to: /Users/friday/SynologyDrive/Personal Drive/stock programs/buy_hold_sell_dashboard/outputs/ceo_report_T_20260512_211141.html



✅  CEO Report saved → /Users/friday/SynologyDrive/Personal Drive/stock programs/buy_hold_sell_dashboard/outputs/ceo_report_T_20260512_211141.html



---
## ✅ CEO Report Complete

**Recommendation: Buy**

Solid fundamentals with more positives than negatives (score: 6.8/10). 5 bullish vs. 3 bearish. Worth a position.

Report saved to: `/Users/friday/SynologyDrive/Personal Drive/stock programs/buy_hold_sell_dashboard/outputs/ceo_report_T_20260512_211141.html`


---
## *(Optional)* Multi-Stock Comparison
Run this cell to compare multiple tickers side by side.

In [ ]:
# ── Multi-stock comparison ─────────────────────────────────────────────────────
# Add tickers here to compare fundamentals side by side
COMPARE_TICKERS = ["AAPL", "MSFT", "GOOG"]  # Edit this list

compare_data = fetch_multiple(COMPARE_TICKERS)
compare_metrics = {}
for t, s in compare_data.items():
    c = MetricsCalculator(stock=s, mode="new")
    compare_metrics[t] = c.compute()

# Print comparison table
from IPython.display import Markdown, display
header = "| Metric | " + " | ".join(COMPARE_TICKERS) + " |"
sep    = "|--------|" + "--------|" * len(COMPARE_TICKERS)
rows = []
metric_defs = [
    ("Overall Score",       lambda m: f"{m.overall_score:.1f}/10"),
    ("Market Cap",          lambda m: fmt_currency(m.q1.market_cap)),
    ("Revenue (TTM)",       lambda m: fmt_currency(m.q1.ttm_revenue)),
    ("Rev Growth (1yr)",    lambda m: fmt_pct(m.q2.revenue_1yr_cagr)),
    ("Operating Margin",    lambda m: fmt_pct(m.q3.operating_margin)),
    ("FCF Yield",           lambda m: fmt_pct(m.q4.fcf_yield)),
    ("Debt/EBITDA",         lambda m: f"{m.q5.debt_to_ebitda:.1f}x" if m.q5.debt_to_ebitda else "N/A"),
    ("P/E",                 lambda m: f"{m.q6.pe_ratio:.1f}x" if m.q6.pe_ratio else "N/A"),
    ("EV/EBITDA",           lambda m: f"{m.q6.ev_ebitda:.1f}x" if m.q6.ev_ebitda else "N/A"),
    ("Margin of Safety",    lambda m: fmt_pct(m.q6.margin_of_safety)),
    ("Dividend Yield",      lambda m: fmt_pct(m.q4.dividend_yield)),
]
for label, fn in metric_defs:
    vals = " | ".join(fn(compare_metrics[t]) for t in COMPARE_TICKERS)
    rows.append(f"| {label} | {vals} |")

table_md = "\n".join([header, sep] + rows)
display(Markdown(f"## Multi-Stock Comparison\n\n{table_md}"))

2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Loading data for AAPL…
2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Data loaded for AAPL — Data as of 2026-05-12 21:11 PDT — last filing 224 days ago
2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Loading data for MSFT…
2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Data loaded for MSFT — Data as of 2026-05-12 21:11 PDT — last filing 316 days ago
2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Loading data for GOOG…
2026-05-12 21:11:42 [INFO] dashboard.data_fetcher — Data loaded for GOOG — Data as of 2026-05-12 21:11 PDT — last filing 132 days ago


## Multi-Stock Comparison

| Metric | AAPL | MSFT | GOOG |
|--------|--------|--------|--------|
| Overall Score | 7.5/10 | 8.0/10 | 7.8/10 |
| Market Cap | $4.33T | $3.03T | $4.65T |
| Revenue (TTM) | $451.44B | $318.27B | $422.50B |
| Rev Growth (1yr) | 6.4% | 14.9% | 15.1% |
| Operating Margin | 32.3% | 46.3% | 36.1% |
| FCF Yield | 2.3% | 2.4% | 1.6% |
| Debt/EBITDA | 0.5x | 0.7x | 0.6x |
| P/E | 35.7x | 24.3x | 29.2x |
| EV/EBITDA | 27.2x | 16.7x | 28.6x |
| Margin of Safety | -191.1% | -456.2% | -418.0% |
| Dividend Yield | 0.4% | 0.9% | 0.2% |

---
## *(Optional)* Manage Threshold Profiles
Save, load, and inspect threshold sets.

In [ ]:
# Save current threshold set for reuse
# tm.save("growth_investor")   # saves to config/user_thresholds/growth_investor.json

# List all saved profiles
profiles = ThresholdManager.list_saved()
print(f"Saved threshold profiles: {profiles}")

# Load a profile
# my_tm = ThresholdManager.load("growth_investor")
# print(my_tm.summary())

Saved threshold profiles: []
